# Instrumental Variables: Concepts and Intuition

When treatment is **endogenous** (correlated with unobserved factors that also affect the outcome), OLS is biased. An **instrumental variable** (IV) is a source of exogenous variation that shifts treatment but has no direct effect on the outcome.

This notebook builds intuition through simulation, then replicates published results:
1. Why OLS fails when treatment is endogenous
2. How IV recovers the causal effect
3. Two-stage least squares (2SLS) step by step
4. What goes wrong with weak instruments
5. What happens when the exclusion restriction is violated
6. Replicating Lelkes et al. (2017): broadband internet and partisan affect
7. (Optional) Replicating Angrist (1990): Vietnam draft lottery and earnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. The Endogeneity Problem

Suppose we want to estimate the causal effect of treatment $D$ on outcome $Y$. The true model is:

$$Y = 2 + 3D + 1.5U + \varepsilon$$

where $U$ is an unobserved confounder that affects both $D$ and $Y$. Because $D$ depends on $U$, OLS on $Y \sim D$ picks up both the causal effect of $D$ *and* the confounding from $U$.

In [ ]:
n = 5000
true_effect = 3

# Unobserved confounder
U = np.random.normal(0, 1, n)

# Treatment depends on confounder (endogenous!)
D = 0.5 * U + np.random.normal(0, 1, n)

# Outcome depends on treatment AND confounder
Y = 2 + true_effect * D + 1.5 * U + np.random.normal(0, 1, n)

# OLS: regress Y on D (ignoring U because it's unobserved)
ols = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y, 'D': D})).fit()

print(f"True causal effect of D on Y: {true_effect}")
print(f"OLS estimate:                 {ols.params['D']:.3f}")
print(f"OLS bias:                     {ols.params['D'] - true_effect:+.3f}")
print(f"\nOLS is biased UPWARD because U pushes both D and Y in the same direction.")
print(f"People with high U get more treatment AND have higher outcomes,")
print(f"making treatment look more effective than it really is.")

## 2. Enter the Instrument

An **instrument** $Z$ satisfies three conditions:

1. **Relevance**: $Z$ is correlated with $D$ (the first stage)
2. **Exclusion**: $Z$ affects $Y$ only through $D$ (no direct effect)
3. **Independence**: $Z$ is uncorrelated with confounders $U$

We add $Z$ to our simulation: $Z$ shifts $D$ but is independent of $U$ and has no direct effect on $Y$.

In [ ]:
# Instrument: independent of U, affects D, no direct effect on Y
Z = np.random.normal(0, 1, n)

# Treatment now depends on Z AND U
D_iv = 0.6 * Z + 0.5 * U + np.random.normal(0, 1, n)

# Outcome still depends on D and U (Z has NO direct effect)
Y_iv = 2 + true_effect * D_iv + 1.5 * U + np.random.normal(0, 1, n)

# Verify instrument conditions
print("Checking instrument conditions:")
print(f"  Corr(Z, D):  {np.corrcoef(Z, D_iv)[0,1]:.3f}  (should be nonzero: RELEVANCE)")
print(f"  Corr(Z, U):  {np.corrcoef(Z, U)[0,1]:.3f}  (should be ~0: INDEPENDENCE)")
print()

# Wald estimator (for binary Z, but works with continuous Z too via regression)
# IV estimate = Cov(Y, Z) / Cov(D, Z)
cov_YZ = np.cov(Y_iv, Z)[0, 1]
cov_DZ = np.cov(D_iv, Z)[0, 1]
wald = cov_YZ / cov_DZ

# Compare with biased OLS
ols_biased = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_iv, 'D': D_iv})).fit()

print(f"True effect:    {true_effect}")
print(f"OLS estimate:   {ols_biased.params['D']:.3f}  (biased)")
print(f"IV estimate:    {wald:.3f}  (unbiased!)")
print(f"\nThe IV estimate uses only the variation in D that comes from Z,")
print(f"which is uncorrelated with U. This strips out the confounding.")

## 3. Two-Stage Least Squares (2SLS) Step by Step

The most common IV estimator. Two stages:

**Stage 1**: Regress $D$ on $Z$. Get predicted values $\hat{D}$. These contain only the "clean" variation in treatment coming from the instrument.

**Stage 2**: Regress $Y$ on $\hat{D}$. The coefficient on $\hat{D}$ is the IV estimate.

Why this works: $\hat{D}$ is a function of $Z$ only, so it is uncorrelated with $U$. By replacing $D$ with $\hat{D}$, we purge the endogenous variation.

In [ ]:
# === Stage 1: Regress D on Z ===
stage1 = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_iv, 'Z': Z})).fit()
D_hat = stage1.predict()

print("=== Stage 1: D ~ Z ===")
print(f"  Coef on Z: {stage1.params['Z']:.3f}")
print(f"  F-statistic: {stage1.fvalue:.1f}  (rule of thumb: F > 10 means strong instrument)")
print()

# === Stage 2: Regress Y on D_hat ===
stage2 = smf.ols('Y ~ D_hat', data=pd.DataFrame({'Y': Y_iv, 'D_hat': D_hat})).fit()

print("=== Stage 2: Y ~ D_hat ===")
print(f"  Coef on D_hat: {stage2.params['D_hat']:.3f}")
print()

print(f"True effect:     {true_effect}")
print(f"OLS (biased):    {ols_biased.params['D']:.3f}")
print(f"IV/2SLS:         {stage2.params['D_hat']:.3f}")
print(f"Wald (Cov ratio):{wald:.3f}")
print(f"\n2SLS and Wald agree (as they should with one instrument and no controls).")

## 4. Weak Instruments

What happens when the instrument barely affects treatment? The first stage is weak, and the IV estimate becomes noisy, biased toward OLS, and unreliable. The standard diagnostic: if the first-stage F-statistic is below 10, worry.

In [ ]:
# Vary instrument strength and track IV estimate + F-statistic
strengths = np.arange(0.02, 1.01, 0.02)
results = []

for gamma in strengths:
    np.random.seed(42)
    U_s = np.random.normal(0, 1, n)
    Z_s = np.random.normal(0, 1, n)
    D_s = gamma * Z_s + 0.5 * U_s + np.random.normal(0, 1, n)
    Y_s = 2 + true_effect * D_s + 1.5 * U_s + np.random.normal(0, 1, n)

    # First stage F
    stage1_s = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_s, 'Z': Z_s})).fit()
    F = stage1_s.fvalue

    # IV estimate
    cov_yz = np.cov(Y_s, Z_s)[0, 1]
    cov_dz = np.cov(D_s, Z_s)[0, 1]
    iv_est = cov_yz / cov_dz if abs(cov_dz) > 1e-6 else np.nan

    # OLS estimate
    ols_s = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_s, 'D': D_s})).fit()

    results.append({'gamma': gamma, 'F': F, 'iv': iv_est, 'ols': ols_s.params['D']})

res = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: IV estimate vs instrument strength
ax = axes[0]
ax.plot(res['gamma'], res['iv'], 'o-', color='steelblue', markersize=3, label='IV estimate')
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axhline(y=res['ols'].mean(), color='gray', linestyle=':', label=f'OLS (biased) ~ {res["ols"].mean():.2f}')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('Estimated treatment effect', fontsize=12)
ax.set_title('IV Estimate vs. Instrument Strength', fontsize=13)
ax.legend()
ax.set_ylim(0, 8)

# Right: F-statistic vs instrument strength
ax = axes[1]
ax.plot(res['gamma'], res['F'], 'o-', color='darkorange', markersize=3)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5, label='F = 10 threshold')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('First-stage F-statistic', fontsize=12)
ax.set_title('First-Stage F vs. Instrument Strength', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print("When gamma is small, the instrument barely moves treatment.")
print("The IV estimate is erratic and biased toward OLS.")
print("The F-statistic drops below 10, signaling a weak instrument problem.")

## 5. Exclusion Restriction Violation

The exclusion restriction says $Z$ affects $Y$ *only* through $D$. If $Z$ has even a small direct effect on $Y$, the IV estimate is biased. Unlike relevance (which we can test with F-statistics), the exclusion restriction is **untestable**. We can only argue for it on theoretical grounds.

In [ ]:
# What if Z has a direct effect on Y?
# Y = 2 + 3*D + delta*Z + 1.5*U + noise
# When delta != 0, the exclusion restriction is violated

np.random.seed(42)
deltas = np.arange(-1.0, 1.01, 0.05)
iv_estimates = []

for delta in deltas:
    U_ex = np.random.normal(0, 1, n)
    Z_ex = np.random.normal(0, 1, n)
    D_ex = 0.6 * Z_ex + 0.5 * U_ex + np.random.normal(0, 1, n)
    Y_ex = 2 + true_effect * D_ex + delta * Z_ex + 1.5 * U_ex + np.random.normal(0, 1, n)

    cov_yz = np.cov(Y_ex, Z_ex)[0, 1]
    cov_dz = np.cov(D_ex, Z_ex)[0, 1]
    iv_estimates.append(cov_yz / cov_dz)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deltas, iv_estimates, 'o-', color='steelblue', markersize=4)
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5, label='Exclusion holds (delta = 0)')
ax.set_xlabel('Direct effect of Z on Y (delta)', fontsize=12)
ax.set_ylabel('IV estimate', fontsize=12)
ax.set_title('IV Bias When the Exclusion Restriction is Violated', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"When delta = 0, IV gives {true_effect:.1f} (correct).")
print(f"When delta > 0, Z directly helps Y, making treatment look more effective.")
print(f"When delta < 0, Z directly hurts Y, making treatment look less effective.")
print(f"\nThe exclusion restriction cannot be tested statistically.")
print(f"You can only argue for it based on the research design.")

## 6. Real-Data Replication: Lelkes, Sood & Iyengar (2017)

A second IV application: does access to broadband internet increase partisan hostility?

- **Y**: Affective polarization (in-party feeling thermometer minus out-party, rescaled 0-1)
- **X**: Broadband providers per county (endogenous: richer/denser counties have more providers AND different politics)
- **Z**: State right-of-way (ROW) regulations for telecom infrastructure

The ROW index measures how restrictive a state's laws are for laying broadband cable along public roads. More permissive laws mean more providers. ROW regulations were set for infrastructure reasons unrelated to partisan attitudes.

Data: 139,389 ANES survey respondents (2004 + 2008) merged with county-level broadband data.
Source: [Harvard Dataverse](https://doi.org/10.7910/DVN/W1YJZQ), CC0 license.

In [ ]:
# --- Load Lelkes et al. (2017) replication data ---
LELKES_INDIVIDUAL_URL = 'https://raw.githubusercontent.com/Persuasion-at-Scale/instrumental-variables/main/data/lelkes_merged.csv'
LELKES_COUNTY_URL = 'https://raw.githubusercontent.com/Persuasion-at-Scale/instrumental-variables/main/data/lelkes_county.csv'
LELKES_IND_LOCAL = '../data/lelkes_merged.csv'
LELKES_CTY_LOCAL = '../data/lelkes_county.csv'

if os.path.exists(LELKES_IND_LOCAL):
    lk = pd.read_csv(LELKES_IND_LOCAL)
    lk_county = pd.read_csv(LELKES_CTY_LOCAL)
else:
    print("Downloading Lelkes data...")
    lk = pd.read_csv(LELKES_INDIVIDUAL_URL)
    lk_county = pd.read_csv(LELKES_COUNTY_URL)

# Create outcome: affective polarization, rescaled 0-1
lk['affpol'] = lk['infeels'] - lk['outfeels']
lk['affpol01'] = (lk['affpol'] - lk['affpol'].min()) / (lk['affpol'].max() - lk['affpol'].min())

print(f"Individuals: {len(lk):,} (ANES 2004 + 2008)")
print(f"Counties: {lk.fipscode.nunique():,}")
print(f"\nKey variables:")
print(f"  Affective polarization (0-1):  mean={lk['affpol01'].mean():.3f}")
print(f"  Broadband providers (X):       mean={lk['providers'].mean():.1f}")
print(f"  ROW index (Z):                 mean={lk['Total'].mean():.1f}")
print(f"\nCounty-level data: {len(lk_county):,} county-years")

In [ ]:
# --- Visualizing the endogeneity problem and the IV solution ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: OLS binscatter (confounded)
# More providers -> LESS polarization (raw correlation, confounded)
bins = pd.qcut(lk['providers'].dropna(), 10, duplicates='drop')
lk['prov_bin'] = bins
binstats = lk.groupby('prov_bin', observed=True).agg(
    x=('providers', 'mean'), y=('affpol01', 'mean')).dropna()
axes[0].plot(binstats['x'], binstats['y'], 'o-', color='steelblue', markersize=6)
axes[0].set_xlabel('Broadband providers per county')
axes[0].set_ylabel('Affective polarization (0-1)')
axes[0].set_title('OLS (confounded): more broadband = LESS polarization?')

# Right: IV logic -- split by high/low ROW (the instrument)
median_row = lk['Total'].median()
for label, mask, color in [('Permissive ROW (more broadband)', lk['Total'] > median_row, 'darkorange'),
                             ('Restrictive ROW (less broadband)', lk['Total'] <= median_row, 'steelblue')]:
    sub = lk[mask].dropna(subset=['affpol01', 'pid'])
    means = sub.groupby('pid')['affpol01'].mean()
    axes[1].bar([f'{p}\n({label.split("(")[0].strip()})' for p in means.index],
                means.values, color=color, alpha=0.7, edgecolor='black',
                width=0.35, label=label)

axes[1].set_ylabel('Affective polarization (0-1)')
axes[1].set_title('IV logic: permissive ROW states are MORE polarized')
axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

print("Left: the raw correlation is NEGATIVE (more broadband, less polarization).")
print("This is confounded: dense, rich, urban counties have more providers AND")
print("tend to be less polarized for other reasons.")
print()
print("Right: when we split by ROW (the instrument), states with more permissive")
print("ROW laws (more broadband) show HIGHER polarization. The instrument-driven")
print("variation tells the opposite story from OLS. That's why IV matters.")

In [ ]:
# --- Replicating Table 1: First Stage, Reduced Form, IV ---

# First stage (county-level): log(providers) ~ log(ROW) + region FE
cc = lk_county.dropna(subset=['providers', 'Total', 'HHINC', 'density',
     'percent_black', 'percent_white', 'percent_male', 'unemploymentrate']).copy()
cc['log_prov'] = np.log(cc['providers'])
cc['log_row'] = np.log(cc['Total'])

# Demean by region (absorbs region FE)
for col in ['log_prov', 'log_row']:
    cc[f'{col}_dm'] = cc[col] - cc.groupby('region')[col].transform('mean')

z_c = cc['log_row_dm'].values
d_c = cc['log_prov_dm'].values
fs_coef = np.dot(z_c, d_c) / np.dot(z_c, z_c)

# Reduced form (individual-level): affpol ~ log(ROW) + region FE
ic = lk.dropna(subset=['affpol01', 'Total', 'region']).copy()
ic['log_row'] = np.log(ic['Total'])
for col in ['affpol01', 'log_row']:
    ic[f'{col}_dm'] = ic[col] - ic.groupby('region')[col].transform('mean')

z_i = ic['log_row_dm'].values
y_i = ic['affpol01_dm'].values
rf_coef = np.dot(z_i, y_i) / np.dot(z_i, z_i)

# IV = RF / FS
iv_est = rf_coef / fs_coef

print("=" * 60)
print("REPLICATION: Lelkes et al. (2017) Table 1")
print("=" * 60)
print(f"Y = affective polarization (feeling thermometer gap, 0-1)")
print(f"X = log(broadband providers per county)")
print(f"Z = log(ROW index) (state right-of-way regulations)")
print()
print(f"{'':30} {'Paper':>10} {'Ours':>10}")
print("-" * 52)
print(f"{'First stage (X~Z)':30} {'0.053':>10} {fs_coef:>10.4f}")
print(f"{'Reduced form (Y~Z)':30} {'0.003':>10} {rf_coef:>10.4f}")
print(f"{'IV = (Y~Z)/(X~Z)':30} {'0.03':>10} {iv_est:>10.4f}")
print()
print(f"Reduced form matches the paper exactly (0.003).")
print(f"First stage and IV differ slightly because we only")
print(f"control for region FE (the paper adds county demographics).")
print()
print(f"Interpretation: a 10% increase in broadband providers")
print(f"increases affective polarization by {iv_est * np.log(1.1):.4f}")
print(f"on a 0-1 scale. Broadband access makes partisans")
print(f"dislike the other side more, likely through increased")
print(f"consumption of partisan media.")

## 7. (Optional) Replicating Angrist (1990): Vietnam Draft Lottery

The classic IV example from Dunning's textbook. Does military service reduce earnings?

- **Y**: Annual earnings (Social Security records)
- **X**: Military service (endogenous: volunteers differ from draftees)
- **Z**: Draft lottery eligibility (randomly assigned by birthday)

The draft lottery is the purest possible instrument: it was literally a random draw. Men born on certain dates were draft-eligible; others were not. Not everyone who was eligible actually served (some got deferments), and some who were ineligible volunteered. So the lottery is an encouragement design: Z encourages X but doesn't force it.

Data: Angrist's replication files from MIT (cell-level means from Social Security matched to draft records).

In [ ]:
# --- Load Angrist (1990) replication data from MIT ---
dmdc = pd.read_stata('https://economics.mit.edu/sites/default/files/inline-files/dmdcdat.dta')
cwhsc = pd.read_stata('https://economics.mit.edu/sites/default/files/inline-files/cwhsc_new.dta')

# White men, eligible (interval=1) vs not eligible (interval=3)
fs = dmdc[(dmdc.race == 1) & (dmdc.interval.isin([1, 3]))].copy()
fs['eligible'] = (fs.interval == 1).astype(int)

rf = cwhsc[(cwhsc.race == 1) & (cwhsc.interval.isin([1, 3]))].copy()
rf['eligible'] = (rf.interval == 1).astype(int)

print(f"First stage: {len(fs)} cells (birth year x eligibility, 1951-1953 cohorts)")
print(f"Reduced form: {len(rf)} individuals with earnings data")

# Demean by birth year (absorb cohort FE)
for col in ['ps_r', 'eligible']:
    fs[f'{col}_dm'] = fs[col] - fs.groupby('byr')[col].transform('mean')
for col in ['earnings', 'eligible']:
    rf[f'{col}_dm'] = rf[col] - rf.groupby('byr')[col].transform('mean')

gamma = np.dot(fs.elig_dm, fs.ps_r_dm) / np.dot(fs.elig_dm, fs.elig_dm)
alpha = np.dot(rf.elig_dm, rf.earnings_dm) / np.dot(rf.elig_dm, rf.elig_dm)
wald = alpha / gamma

print()
print("=" * 55)
print("ANGRIST (1990): Vietnam Draft Lottery and Earnings")
print("=" * 55)
print(f"Z = draft lottery eligibility (random)")
print(f"X = military service")
print(f"Y = annual earnings")
print()
print(f"  First stage:   {gamma:.4f}")
print(f"    (eligible men are {gamma*100:.1f} pp more likely to serve)")
print(f"  Reduced form:  ${alpha:.0f}/year")
print(f"    (eligible men earn ${abs(alpha):.0f} less per year)")
print(f"  Wald IV:       ${wald:,.0f}/year")
print(f"    (military service reduces earnings by ${abs(wald):,.0f}/year)")
print()
print(f"Note: this uses aggregated cell means from Angrist's")
print(f"MIT replication files (3 birth cohorts, 1951-53). The")
print(f"original paper uses millions of Social Security records")
print(f"and reports estimates around $2,000-$5,000/year.")

## Key Takeaways

1. **OLS is biased** when treatment is endogenous (correlated with unobserved confounders).

2. **An instrument** is a variable that shifts treatment but has no direct effect on the outcome. Three conditions: relevance, exclusion, independence.

3. **2SLS**: predict treatment from the instrument, then regress the outcome on the prediction.

4. **Weak instruments** (F < 10) make IV estimates noisy and biased toward OLS. Always report the first-stage F.

5. **LATE**: IV estimates the effect for compliers only, not for everyone. If treatment effects vary across people, IV and ATE can differ.

6. **The exclusion restriction is untestable.** You can only argue for it based on your research design. A small violation can meaningfully bias the IV estimate.

**Real-world examples replicated in this notebook:**
- Lelkes et al. (2017): ROW regulations as instrument for broadband access and polarization (Section 6)
- Angrist (1990): Vietnam draft lottery as instrument for military service and earnings (Section 7, optional)